In [2]:
import json
import xml.etree.ElementTree as ET
from typing import Dict, Any
import geopandas as gpd
from supabase import create_client, Client
import os
from dotenv import load_dotenv

load_dotenv()

# supabase: Client = create_client(url, key)

In [23]:
class KMLtoSupabasePipeline:
    def __init__(self, supabase_url: str, supabase_key: str, project_code: str):
        """Initialize the pipeline with Supabase credentials"""
        self.supabase: Client = create_client(supabase_url, supabase_key)
        self.project_code = project_code 
        
    def kml_to_geojson(self, kml_file_path: str) -> Dict[str, Any]:
        """Convert KML file to GeoJSON format"""
        try:
            # Read KML file using geopandas
            gdf = gpd.read_file(kml_file_path, driver='KML')
            
            # Convert to GeoJSON
            geojson_data = json.loads(gdf.to_json())
            return geojson_data
            
        except Exception as e:
            print(f"Error converting KML to GeoJSON: {str(e)}")
            return None
        
    def get_project_id(self) -> str:
        """Retrieve project_id from projects table using project_code"""
        try:
            response = (self.supabase.table("projects")
                       .select("id")
                       .eq("project_code", self.project_code)
                       .execute())
            if response.data and len(response.data) > 0:
                return response.data[0]["id"]
            print(f"No project found with project_code: {self.project_code}")
            return None
        except Exception as e:
            print(f"Error retrieving project_id: {str(e)}")
            return None
        
    def upload_to_supabase(self, geojson_data: Dict[str, Any], project_id: str, table_name: str = "geo_data"):
        """Upload GeoJSON to Supabase with project_code matching"""
        try:
            # Prepare data for Supabase
            data = {
                "project_id": project_id,
                "geometry": geojson_data,
            }

            # Check if project_code exists
            existing = (self.supabase.table(table_name)
                       .select("project_id")
                       .eq("project_id", project_id)
                       .execute())
            
            if len(existing.data) > 0:
                # Update existing record
                response = (self.supabase.table(table_name)
                          .update(data)
                          .eq("project_id", project_id)
                          .execute())
            else:
                # Insert new record
                response = self.supabase.table(table_name).insert(data).execute()
                
            return response
            
        except Exception as e:
            print(f"Error uploading to Supabase: {str(e)}")
            return None

    def process_file(self, kml_file_path: str, table_name: str = "geo_data", ):
        """Main pipeline method to process KML file and upload to Supabase"""
        # Step 1: Convert KML to GeoJSON
        geojson_data = self.kml_to_geojson(kml_file_path)
        if not geojson_data:
            return False
        
        # Step 2: Get project_id for the project_code
        project_id = self.get_project_id()
        if not project_id:
            return False

        # Step 2: Upload to Supabase
        response = self.upload_to_supabase(geojson_data, project_id, table_name)
        
        if response:
            print(f"Successfully processed {kml_file_path} with project_code: {self.project_code}")
            return True
        return False

In [ ]:
KML_FILE_PATH = "/Users/beckyxu/Downloads/PadangTikar_ElegibleArea_stableforest_06_16_PA.kml"

url: str = os.getenv("SUPABASE_URL", "")
key: str = os.getenv("SUPABASE_KEY", "")

# Initialize pipeline
pipeline = KMLtoSupabasePipeline(url, key, "3226")

# Process the file
success = pipeline.process_file(KML_FILE_PATH)

if success:
    print("Pipeline completed successfully")
else:
    print("Pipeline failed")

In [ ]:
gdf = gpd.read_file(KML_FILE_PATH, driver='KML')

# Convert to GeoJSON
geojson_data = json.loads(gdf.to_json())
geojson_data